In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from ase.calculators.espresso import Espresso, EspressoProfile
from ase.io import read
from ase.units import GPa, Ry


In [2]:
atoms = read("PbO.cif")
orig_cell = atoms.get_cell().copy()
orig_scaled_positions = atoms.get_scaled_positions().copy()
orig_volume = atoms.get_volume()

print(orig_cell)
print(orig_volume)
print(orig_scaled_positions)

Cell([3.9744, 3.9744, 5.022])
79.32678561792001
[[0.     0.5    0.2351]
 [0.5    0.     0.7649]
 [0.     0.     0.    ]
 [0.5    0.5    0.    ]]


In [8]:
pseudo_dir = str(Path("src").resolve())
pseudopotentials = {"Pb": "Pb.pbe-dn-kjpaw_psl.0.2.2.UPF",
                    "O": "O.pbe-n-kjpaw_psl.0.1.UPF"}

input_data = {
    'control': {
        'calculation': 'vc-relax',       # Options: 'scf', 'relax', 'vc-relax'
        'prefix': 'FeAl',              # Prefix for input/output files
        'verbosity': 'low',
        "pseudo_dir": pseudo_dir,  # Directory for pseudopotentials
        'tprnfor': True,
        'tstress': True,
        'outdir': '.',          # Directory for scratch files
    },
    'system': {
        'ecutwfc': 60.0,            # Wavefunction cutoff in Ry
        'ecutrho': 240.0,          # Charge density cutoff in Ry
        'occupations': 'smearing',
        'smearing': 'mv',
        'degauss': 0.005,
    },
    'electrons': {
        'conv_thr': 1e-08,         # Convergence threshold
        'mixing_beta': 0.7,
    },
    'ions':{
        'ion_dynamics': 'bfgs'
    },
    'cell':{
        'cell_dynamics': 'bfgs',
        'press': 0.0,
        'press_conv_thr': 0.5
    }
}
kpts = (5,5,5)  # k-point mesh

In [9]:
profile = EspressoProfile(command='pw.x', pseudo_dir=pseudo_dir)
atoms.calc = Espresso(profile=profile, pseudopotentials=pseudopotentials,
                    kpts=kpts, input_data=input_data,
                    directory=Path("out"))
atoms.get_potential_energy()


KeyboardInterrupt: 